# Thí nghiệm đối chứng Bank Marketing

Dùng split Parquet của nhóm và ngưỡng 0,5 cho các phép so sánh. Cột PR-AUC trong kết quả là Average Precision (AP) theo hàm evaluate dùng chung. Accuracy được trình bày để đối chiếu với baseline luôn dự đoán no.

In [ ]:
import io
import json
import sys
import warnings
from contextlib import redirect_stdout
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as SamplingPipeline
from imblearn.under_sampling import RandomUnderSampler
from IPython.display import Image, display
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root))
from src.metrics import evaluate

results_dir = root / "reports" / "tv4"
figures_dir = root / "reports" / "figures"
results_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)
data_dir = root / "data" / "processed"

def load_split(name):
    main = pd.read_parquet(data_dir / f"{name}.parquet")
    extended = pd.read_parquet(data_dir / f"{name}_with_duration.parquet")
    if "duration" not in extended or not extended.drop(columns="duration").equals(main):
        raise ValueError(f"{name}: the two variants must differ only by duration.")
    if "y" not in main or set(main["y"].unique()) != {0, 1} or not np.isfinite(extended).all(axis=None):
        raise ValueError(f"{name}: expected finite numeric columns and labels 0/1.")
    return main.drop(columns="y"), extended.drop(columns="y"), main["y"]

X_train, X_train_duration, y_train = load_split("train")
X_test, X_test_duration, y_test = load_split("test")
if not X_train.columns.equals(X_test.columns) or not X_train_duration.columns.equals(X_test_duration.columns):
    raise ValueError("Train and test feature names or order differ.")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f"Train: {len(y_train):,} | Test: {len(y_test):,} | Test yes: {y_test.mean():.2%}")

## Logistic Regression và biến vĩ mô

Kiểm tra tương quan trên train giữa euribor3m, nr.employed và emp.var.rate. So sánh ứng viên giữ đủ biến với ứng viên bỏ euribor3m và nr.employed, giữ emp.var.rate làm đại diện. Chọn bộ cột theo AP trung bình qua 5 fold của train. StandardScaler nằm trong pipeline nên fit lại trên phần huấn luyện của từng fold.

In [ ]:
macro_cols = ["euribor3m", "nr.employed", "emp.var.rate"]
macro_corr = X_train[macro_cols].corr()
display(macro_corr.round(4))

logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
features = {
    "full": list(X_train.columns),
    "reduced": [column for column in X_train.columns if column not in macro_cols[:2]],
}

def cv_ap(estimator, frame):
    return cross_val_score(estimator, frame, y_train, cv=cv, scoring="average_precision", error_score="raise")

macro_runs = pd.DataFrame([
    {"features": key, "columns": len(columns), "cv_ap_mean": cv_ap(logistic, X_train[columns]).mean()}
    for key, columns in features.items()
])
selected = macro_runs.loc[macro_runs["cv_ap_mean"].idxmax(), "features"]
macro_runs["selected"] = macro_runs["features"].eq(selected)
macro_runs.to_csv(results_dir / "macro_comparison.csv", index=False, float_format="%.8f")
display(macro_runs.round(4))

## Baseline và thí nghiệm duration

DummyClassifier luôn trả về no trên split hiện tại. Với Logistic Regression và hai mô hình cây, giữ nguyên cấu hình khi thêm duration vào bộ đặc trưng. Decision Tree và Random Forest dùng tham số mặc định của scikit-learn cùng seed 42 để đối chiếu trong khi nhóm chốt cấu hình của TV2.

In [ ]:
def score(model, frame, name):
    probability = model.predict_proba(frame)[:, 1]
    with redirect_stdout(io.StringIO()), warnings.catch_warnings():
        warnings.simplefilter("ignore", UndefinedMetricWarning)
        metrics = evaluate(y_test, y_prob=probability, model_name=name, plot_cm=False)
    metrics["Accuracy"] = (y_test.to_numpy() == (probability >= 0.5)).mean()
    return metrics, probability

# ponytail: use the default trees until TV2 publishes its selected configuration.
estimators = {
    "Logistic Regression": logistic,
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
}

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
metrics, _ = score(dummy, X_test, "Luôn dự đoán no")
rows = [{**metrics, "with_duration": False}]
fitted, probabilities = {}, {}
for name, estimator in estimators.items():
    for keep_duration in (False, True):
        train, test = (X_train_duration, X_test_duration) if keep_duration else (X_train, X_test)
        if name == "Logistic Regression":
            columns = features[selected] + (["duration"] if keep_duration else [])
            train, test = train[columns], test[columns]
        model = clone(estimator).fit(train, y_train)
        metrics, probability = score(model, test, name)
        rows.append({**metrics, "with_duration": keep_duration})
        fitted[name, keep_duration] = model
        probabilities[name, keep_duration] = probability

comparison = pd.DataFrame(rows)
comparison.to_csv(results_dir / "model_metrics.csv", index=False, float_format="%.8f")
display(comparison.round(4))

## Xử lý mất cân bằng

Dùng Random Forest bỏ duration và cùng các fold phân tầng. So sánh dữ liệu gốc, class_weight, RandomOverSampler và RandomUnderSampler. Hai bộ lấy mẫu chạy trong imblearn.Pipeline ở từng fold train. Validation và test giữ phân bố tự nhiên.

In [ ]:
samplers = {
    "original": "passthrough",
    "class_weight": "passthrough",
    "oversampling": RandomOverSampler(random_state=RANDOM_STATE),
    "undersampling": RandomUnderSampler(random_state=RANDOM_STATE),
}
rows = []
for method, sampler in samplers.items():
    rf = clone(estimators["Random Forest"])
    if method == "class_weight":
        rf.set_params(class_weight="balanced")
    pipeline = SamplingPipeline([("sampler", sampler), ("model", rf)])
    ap = cv_ap(pipeline, X_train)
    if method == "original":
        metrics = comparison.loc[
            comparison["Model"].eq("Random Forest") & ~comparison["with_duration"]
        ].iloc[0].drop(labels="with_duration").to_dict()
    else:
        metrics, _ = score(pipeline.fit(X_train, y_train), X_test, "Random Forest")
    rows.append({**metrics, "method": method, "cv_ap_mean": ap.mean(), "cv_ap_std": ap.std(ddof=1)})

imbalance = pd.DataFrame(rows)
imbalance.to_csv(results_dir / "imbalance_metrics.csv", index=False, float_format="%.8f")
display(imbalance.round(4))

assert len(comparison) == 7 and len(imbalance) == 4
assert set(comparison["Threshold"]) == {0.5}
for table in (comparison, imbalance):
    assert (table[["TN", "FP", "FN", "TP"]].sum(axis=1) == len(y_test)).all()
    assert (table["FN"] + table["TP"] == y_test.sum()).all()
assert comparison.iloc[0]["TP"] == comparison.iloc[0]["FP"] == 0

## Hình dùng trong báo cáo

Các hình ROC, PR và ma trận nhầm lẫn dùng bộ đặc trưng bỏ duration. Đường tham chiếu PR biểu thị tỷ lệ đăng ký trên test. Hình xuất nền trắng, palette colorblind và 300 dpi.

In [ ]:
sns.set_theme(
    style="whitegrid", palette="colorblind",
    rc={"font.family": "DejaVu Sans", "savefig.dpi": 300, "axes.spines.top": False, "axes.spines.right": False},
)
colors = dict(zip(estimators, sns.color_palette("colorblind", 3), strict=True))

def show(fig, filename):
    path = figures_dir / filename
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    display(Image(filename=path, width=920))

main = comparison.loc[~comparison["with_duration"]].set_index("Model")
fig, (roc_ax, pr_ax) = plt.subplots(1, 2, figsize=(10, 4.2), layout="constrained")
roc_ax.plot([0, 1], [0, 1], "--", color="0.55", label="Ngẫu nhiên")
pr_ax.axhline(y_test.mean(), linestyle="--", color="0.55", label=f"Tỷ lệ yes {y_test.mean():.1%}")
for name, line in zip(estimators, ("-", "-.", ":"), strict=True):
    probability = probabilities[name, False]
    fpr, tpr, _ = roc_curve(y_test, probability)
    precision, recall, _ = precision_recall_curve(y_test, probability)
    roc_ax.plot(fpr, tpr, line, color=colors[name], label=f"{name} ({main.at[name, 'ROC-AUC']:.4f})")
    pr_ax.step(recall, precision, where="post", linestyle=line, color=colors[name],
               label=f"{name} ({main.at[name, 'PR-AUC']:.4f})")
roc_ax.set(title="ROC · AUC", xlabel="Tỷ lệ dương tính giả", ylabel="Tỷ lệ dương tính thật")
pr_ax.set(title="Precision–Recall · AP", xlabel="Recall", ylabel="Precision")
for ax in (roc_ax, pr_ax):
    ax.set(xlim=(0, 1), ylim=(0, 1.02))
    ax.legend(loc="lower right" if ax is roc_ax else "upper right", fontsize=8)
show(fig, "tv4_roc_pr.png")

fig, axes = plt.subplots(1, 4, figsize=(12.4, 3.5), layout="constrained")
for ax, (name, row) in zip(axes, main.iterrows(), strict=True):
    counts = np.array([[row.TN, row.FP], [row.FN, row.TP]], dtype=int)
    ConfusionMatrixDisplay(counts, display_labels=["no", "yes"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d",
        im_kw={"vmin": 0, "vmax": int(y_test.value_counts().max())},
    )
    ax.set(title=name, xlabel="Dự đoán", ylabel="Thực tế")
    ax.grid(False)
fig.suptitle("Ma trận nhầm lẫn ở ngưỡng 0,5")
show(fig, "tv4_confusion_matrices.png")

## Độ quan trọng của đặc trưng

Dùng mức giảm độ hỗn tạp từ các mô hình cây đã fit trên train bỏ duration. Cộng đóng góp của các cột one-hot theo biến gốc để bảng có cùng cách đọc giữa hai mô hình.

In [ ]:
metadata = json.loads((root / "models" / "feature_metadata.json").read_text(encoding="utf-8"))
groups = [
    next((name for name in metadata["categorical_columns"] if column.startswith(name + "_")), column)
    for column in X_train.columns
]
tables = []
tree_names = ("Decision Tree", "Random Forest")
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), layout="constrained", sharex=True)
for ax, name in zip(axes, tree_names, strict=True):
    values = fitted[name, False].feature_importances_
    importance = pd.Series(values, index=groups).groupby(level=0).sum().sort_values(ascending=False)
    assert np.isclose(importance.sum(), 1)
    tables.append(importance.rename_axis("feature").reset_index(name="importance").assign(model=name))
    top = importance.head(8).iloc[::-1]
    ax.barh(top.index, top, color=colors[name])
    ax.set(title=name, xlabel="Độ quan trọng theo mức giảm độ hỗn tạp")
    ax.grid(axis="y", visible=False)
pd.concat(tables).to_csv(results_dir / "feature_importance.csv", index=False, float_format="%.8f")
show(fig, "tv4_feature_importance.png")

## Nhóm khách ưu tiên gọi

Minh họa ngân sách gọi 10% tập test bằng điểm Random Forest bỏ duration. Tỷ lệ đăng ký quan sát được cho nhóm điểm cao và nhóm từng có chiến dịch thành công giúp diễn giải bảng độ quan trọng.

In [ ]:
print("Số mức xác suất dự đoán trên test", {
    name: np.unique(probabilities[name, False]).size for name in tree_names
})
limit = int(np.ceil(0.10 * len(y_test)))
order = np.argsort(-probabilities["Random Forest", False], kind="stable")
cohorts = {
    "all_test": y_test,
    "random_forest_top_10_percent": y_test.iloc[order[:limit]],
    "previous_campaign_success": y_test[X_test["poutcome_success"].eq(1)],
}
priority = pd.DataFrame([
    {"group": name, "customers": len(labels), "subscriptions": int(labels.sum()),
     "subscription_rate": labels.mean()}
    for name, labels in cohorts.items()
])
priority.to_csv(results_dir / "outreach_priorities.csv", index=False, float_format="%.8f")
display(priority.round(4))

## Benchmark Spark và scikit-learn

Chạy benchmark_tv4.py cho mỗi engine trên cùng train.parquet. Bảng tổng hợp và hai biểu đồ dùng trung bình các lần chạy. Các mức nhân bản chỉ mô phỏng tải huấn luyện.

In [ ]:
benchmark_path = results_dir / "benchmark_runs.csv"
if benchmark_path.is_file():
    runs = pd.read_csv(benchmark_path)
    assert not runs.duplicated(["engine", "factor", "run"]).any()
    assert (runs["train_rows"] == len(y_train) * runs["factor"]).all()
    benchmark = runs.groupby(["engine", "factor", "train_rows"], as_index=False).agg(
        runs=("run", "size"),
        fit_seconds_mean=("fit_seconds", "mean"),
        fit_seconds_std=("fit_seconds", "std"),
        peak_rss_mib_mean=("peak_rss_mib", "mean"),
        peak_rss_mib_max=("peak_rss_mib", "max"),
    )
    benchmark.to_csv(results_dir / "benchmark_summary.csv", index=False, float_format="%.6f")
    display(benchmark.round(3))

    engine_styles = {"scikit-learn": ("o-", "#0173b2"), "Spark": ("s--", "#de8f05")}
    for metric, label, filename in (
        ("fit_seconds_mean", "Thời gian fit trung bình (giây)", "tv4_benchmark_time.png"),
        ("peak_rss_mib_mean", "RAM đỉnh trung bình (MiB)", "tv4_benchmark_ram.png"),
    ):
        fig, ax = plt.subplots(figsize=(7, 3.8), layout="constrained")
        for engine, subset in benchmark.groupby("engine"):
            subset = subset.sort_values("train_rows")
            style, color = engine_styles[engine]
            ax.plot(subset["train_rows"], subset[metric], style, color=color, label=engine)
        sizes = sorted(benchmark["train_rows"].unique())
        ax.set(xscale="log", xlabel="Số dòng huấn luyện (thang log)", ylabel=label)
        ax.set_xticks(sizes, [f"{size:,}".replace(",", ".") for size in sizes])
        ax.legend()
        show(fig, filename)
else:
    print("Chạy benchmark_tv4.py để tạo benchmark_runs.csv.")